## HDC (QSVM)

In [ ]:
!pip install qiskit==1.0.0
!pip install qiskit-algorithms
!pip install qiskit-machine-learning
!pip install qiskit-aer
!pip install qiskit-ibm-runtime==0.22.0

In [1]:
import numpy as np
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
import sobol_seq  # Import Sobol sequence library

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 4  # Increased dimensionality for better representation

# Generate Sobol sequences for the projection matrix
print("Generating Sobol sequence projection...")
sobol_seq_generator = sobol_seq.i4_sobol_generate(X_train.shape[1], D)  # Generating Sobol sequences with the right shape
proj = sobol_seq_generator.T  # Transpose to match the shape (D, number_of_features)

def project_data(data, proj):
    return np.dot(data, proj)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, proj)

feature_map = ZZFeatureMap(feature_dimension=D, reps=3, entanglement='full')

sampler = Sampler()
fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

qsvc = QSVC(quantum_kernel=quantum_kernel)
qsvc.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = qsvc.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, proj)
# Classify test data using the trained QSVC classifier
y_pred = qsvc.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in qsvc.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QSVM) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Classical-Quantum HDC(QSVM) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Classical-Quantum HDC(QSVM) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sobol sequence projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_12360\857694218.py:54: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()


Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: 87.50%
Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: 81.82%
Hybrid Classical-Quantum HDC(QSVM) Training Time: 6.6608 seconds
Hybrid Classical-Quantum HDC(QSVM) Inference Time: 4.7050 seconds
Hybrid Classical-Quantum HDC(QSVM) Model Memory Required: 0.0051 MB


## IBM AerSimulator (HDC(QSVM))

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import PauliFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from sklearn.preprocessing import StandardScaler

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV1 as Sampler,
    EstimatorOptions
)
from scipy.stats import qmc

QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="4d198f821adff3ba7447f905aefdc3d1ec3b04c387bff8ac4f0a01da6b7315bf036c83aa4d24564c9f823cfae6fb86ad6b8bff598ca7c76590ce2c1cb4ebbd10",
    set_as_default=True,
    overwrite=True,
)

# Load dataset
def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 4

# Generate Sobol sequence
#print("Generating Sobol sequence projection...")
#sobol_sampler = qmc.Sobol(d=X_train.shape[1], scramble=True)
#proj = sobol_sampler.random_base2(m=int(np.log2(D)))  # Generate D-dimensional Sobol sequence
print("Generating Sobol sequence projection...")
sobol_seq_generator = sobol_seq.i4_sobol_generate(X_train.shape[1], D)  # Generating Sobol sequences with the right shape
proj = sobol_seq_generator.T  # Transpose to match the shape (D, number_of_features)

def project_data(data, proj):
    return np.dot(data, proj)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, proj)

feature_map = PauliFeatureMap(feature_dimension=X_train.shape[1], reps=3, entanglement='full')

aer_sim = AerSimulator()
sampler = Sampler(backend=aer_sim)

fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

start_train_time = time.time()

qsvc = QSVC(quantum_kernel=quantum_kernel)
qsvc.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = qsvc.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, proj)
# Classify test data using the trained QSVC classifier
y_pred = qsvc.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in qsvc.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QSVM) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Classical-Quantum HDC(QSVM) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Classical-Quantum HDC(QSVM) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sobol sequence projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_12360\810474598.py:77: DeprecationWarning: The Sampler and Estimator V1 primitives have been deprecated as of qiskit-ibm-runtime 0.23.0 and will be removed no sooner than 3 months after the release date. Please use the V2 Primitives. See the `V2 migration guide <https://docs.quantum.ibm.com/api/migration-guides/v2-primitives>`_. for more details
  sampler = Sampler(backend=aer_sim)


Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: 85.00%
Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: 81.82%
Hybrid Classical-Quantum HDC(QSVM) Training Time: 6.6316 seconds
Hybrid Classical-Quantum HDC(QSVM) Inference Time: 4.2528 seconds
Hybrid Classical-Quantum HDC(QSVM) Model Memory Required: 0.0051 MB


## IBM Sherbrooke Hardware HDC(QSVM)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import PauliFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from sklearn.preprocessing import StandardScaler

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV1 as Sampler,
    EstimatorOptions
)
from scipy.stats import qmc

QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="4d198f821adff3ba7447f905aefdc3d1ec3b04c387bff8ac4f0a01da6b7315bf036c83aa4d24564c9f823cfae6fb86ad6b8bff598ca7c76590ce2c1cb4ebbd10",
    set_as_default=True,
    overwrite=True,
)

# Load dataset
def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 4

# Generate Sobol sequence
#print("Generating Sobol sequence projection...")
#sobol_sampler = qmc.Sobol(d=X_train.shape[1], scramble=True)
#proj = sobol_sampler.random_base2(m=int(np.log2(D)))  # Generate D-dimensional Sobol sequence

print("Generating Sobol sequence projection...")
sobol_seq_generator = sobol_seq.i4_sobol_generate(X_train.shape[1], D)  # Generating Sobol sequences with the right shape
proj = sobol_seq_generator.T  # Transpose to match the shape (D, number_of_features)


def project_data(data, proj):
    return np.dot(data, proj)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, proj)

feature_map = PauliFeatureMap(feature_dimension=X_train.shape[1], reps=3, entanglement='full')

service = QiskitRuntimeService()
backend = service.backend("ibm_sherbrooke")

fake_backend = AerSimulator.from_backend(backend)

sampler = Sampler(backend=fake_backend)

fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)

start_train_time = time.time()

qsvc = QSVC(quantum_kernel=quantum_kernel)
qsvc.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = qsvc.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, proj)
# Classify test data using the trained QSVC classifier
y_pred = qsvc.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in qsvc.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QSVM) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Classical-Quantum HDC(QSVM) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Classical-Quantum HDC(QSVM) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sobol sequence projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_12360\1489640937.py:81: DeprecationWarning: The class ``qiskit.providers.models.backendconfiguration.QasmBackendConfiguration`` is deprecated as of qiskit 1.2. It will be removed in the 2.0 release. The models in ``qiskit.providers.models`` are part of the deprecated `BackendV1` workflow and no longer necessary for `BackendV2`. If a user workflow requires these representations it likely relies on deprecated functionality and should be updated to use `BackendV2`.
  fake_backend = AerSimulator.from_backend(backend)
C:\Users\abuka\AppData\Local\Temp\ipykernel_12360\1489640937.py:83: DeprecationWarning: The Sampler and Estimator V1 primitives have been deprecated as of qiskit-ibm-runtime 0.23.0 and will be removed no sooner than 3 months after the release date. Please use the V2 Primitives. See the `V2 migration guide <https://docs.quantum.ibm.com/api/migration-guides/v2-primitives>`_. for more details
  sampler = Sampler(backend=fake_backend)


Hybrid Classical-Quantum HDC(QSVM) Train Accuracy: 95.00%
Hybrid Classical-Quantum HDC(QSVM) Test Accuracy: 90.91%
Hybrid Classical-Quantum HDC(QSVM) Training Time: 8.5743 seconds
Hybrid Classical-Quantum HDC(QSVM) Inference Time: 6.3127 seconds
Hybrid Classical-Quantum HDC(QSVM) Model Memory Required: 0.0051 MB


## HDC(QNN)

In [7]:
import numpy as np
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.circuit.library import QNNCircuit
import sobol_seq  # Import Sobol sequence library

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 5  # Increased dimensionality for better representation

# Create Sobol sequences for the projection matrix
print("Generating Sobol sequence projection...")
# Generate Sobol sequences for the number of dimensions needed
sobol_seq_generator = sobol_seq.i4_sobol_generate(X_train.shape[1], D)  # Generating Sobol sequences with the right shape
proj = sobol_seq_generator.T  # Transpose to match the shape (D, number_of_features)

def project_data(data, proj):
    return np.dot(data, proj)

# Project training and test data to hyperdimensional space
X_train_proj = project_data(X_train, proj)
X_test_proj = project_data(X_test, proj)

# Define Quantum Circuit with increased number of reps and different ansatz
feature_map = ZZFeatureMap(feature_dimension=D, reps=2, entanglement='full')
qc = QNNCircuit(ansatz=RealAmplitudes(D, reps=3))  # Increased reps for more expressivity

def parity(x):
    return "{:b}".format(x).count("1") % 2

output_shape = 2

sampler_qnn = SamplerQNN(
    circuit=qc,
    interpret=parity,
    output_shape=output_shape,
)

# Use L-BFGS-B optimizer with adjusted parameters for better performance
sampler_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn, optimizer=L_BFGS_B(maxiter=300)
)

# Fit classifier to data
start_train_time = time.time()
sampler_classifier.fit(X_train_proj, y_train)
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = sampler_classifier.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QNN) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
y_pred = sampler_classifier.predict(X_test_proj)
end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QNN) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in sampler_classifier.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QNN) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Classical-Quantum HDC(QNN) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Classical-Quantum HDC(QNN) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sobol sequence projection...
Hybrid Classical-Quantum HDC(QNN) Train Accuracy: 92.50%
Hybrid Classical-Quantum HDC(QNN) Test Accuracy: 45.45%
Hybrid Classical-Quantum HDC(QNN) Training Time: 600.3140 seconds
Hybrid Classical-Quantum HDC(QNN) Inference Time: 0.0678 seconds
Hybrid Classical-Quantum HDC(QNN) Model Memory Required: 0.0018 MB


## IBM AerSimulator (HDC(QNN))

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import PauliFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.circuit.library import QNNCircuit

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV1 as Sampler,
    EstimatorOptions
)
from scipy.stats import qmc

QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="4d198f821adff3ba7447f905aefdc3d1ec3b04c387bff8ac4f0a01da6b7315bf036c83aa4d24564c9f823cfae6fb86ad6b8bff598ca7c76590ce2c1cb4ebbd10",
    set_as_default=True,
    overwrite=True,
)

# Load dataset
def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 5

# Generate Sobol sequence
print("Generating Sobol sequence projection...")
#sobol_sampler = qmc.Sobol(d=X_train.shape[1], scramble=True)
#proj = sobol_sampler.random_base2(m=int(np.log2(D)))  # Generate D-dimensional Sobol sequence
sobol_seq_generator = sobol_seq.i4_sobol_generate(X_train.shape[1], D)  # Generating Sobol sequences with the right shape
proj = sobol_seq_generator.T  # Transpose to match the shape (D, number_of_features)


def project_data(data, proj):
    return np.dot(data, proj)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, proj)

feature_map = PauliFeatureMap(feature_dimension=X_train.shape[1], reps=3, entanglement='full')

aer_sim = AerSimulator()
sampler = Sampler(backend=aer_sim)

qc = QNNCircuit(ansatz=RealAmplitudes(D, reps=3))  # Increased reps for more expressivity

def parity(x):
    return "{:b}".format(x).count("1") % 2

output_shape = 2

sampler_qnn = SamplerQNN(
    circuit=qc,
    interpret=parity,
    output_shape=output_shape,
    sampler = sampler
)

# Use L-BFGS-B optimizer with adjusted parameters for better performance
sampler_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn, optimizer=L_BFGS_B(maxiter=300)
)

# Fit classifier to data
start_train_time = time.time()
sampler_classifier.fit(X_train_proj, y_train)
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = sampler_classifier.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QNN) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
y_pred = sampler_classifier.predict(X_test_proj)
end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QNN) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in sampler_classifier.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QNN) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Classical-Quantum HDC(QNN) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Classical-Quantum HDC(QNN) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sobol sequence projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_14172\4103636239.py:86: DeprecationWarning: The Sampler and Estimator V1 primitives have been deprecated as of qiskit-ibm-runtime 0.23.0 and will be removed no sooner than 3 months after the release date. Please use the V2 Primitives. See the `V2 migration guide <https://docs.quantum.ibm.com/api/migration-guides/v2-primitives>`_. for more details
  sampler = Sampler(backend=aer_sim)


Hybrid Classical-Quantum HDC(QNN) Train Accuracy: 90.00%
Hybrid Classical-Quantum HDC(QNN) Test Accuracy: 54.55%
Hybrid Classical-Quantum HDC(QNN) Training Time: 235.9445 seconds
Hybrid Classical-Quantum HDC(QNN) Inference Time: 0.2014 seconds
Hybrid Classical-Quantum HDC(QNN) Model Memory Required: 0.0018 MB


## IBM Sherbrooke Hardware HDC(QNN)

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import PauliFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.circuit.library import QNNCircuit

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV1 as Sampler,
    EstimatorOptions
)
from scipy.stats import qmc

QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="4d198f821adff3ba7447f905aefdc3d1ec3b04c387bff8ac4f0a01da6b7315bf036c83aa4d24564c9f823cfae6fb86ad6b8bff598ca7c76590ce2c1cb4ebbd10",
    set_as_default=True,
    overwrite=True,
)

# Load dataset
def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X[:51], y[:51]

# Shuffle dataset
def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

# Load and shuffle the dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 5

# Generate Sobol sequence
print("Generating Sobol sequence projection...")
#sobol_sampler = qmc.Sobol(d=X_train.shape[1], scramble=True)
#proj = sobol_sampler.random_base2(m=int(np.log2(D)))  # Generate D-dimensional Sobol sequence
sobol_seq_generator = sobol_seq.i4_sobol_generate(X_train.shape[1], D)  # Generating Sobol sequences with the right shape
proj = sobol_seq_generator.T  # Transpose to match the shape (D, number_of_features)

def project_data(data, proj):
    return np.dot(data, proj)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, proj)

feature_map = PauliFeatureMap(feature_dimension=X_train.shape[1], reps=3, entanglement='full')

service = QiskitRuntimeService()
backend = service.backend("ibm_sherbrooke")

fake_backend = AerSimulator.from_backend(backend)
sampler = Sampler(backend=fake_backend)

qc = QNNCircuit(ansatz=RealAmplitudes(D, reps=3))  # Increased reps for more expressivity

def parity(x):
    return "{:b}".format(x).count("1") % 2

output_shape = 2

sampler_qnn = SamplerQNN(
    circuit=qc,
    interpret=parity,
    output_shape=output_shape,
    sampler = sampler
)

# Use L-BFGS-B optimizer with adjusted parameters for better performance
sampler_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn, optimizer=L_BFGS_B(maxiter=300)
)

# Fit classifier to data
start_train_time = time.time()
sampler_classifier.fit(X_train_proj, y_train)
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = sampler_classifier.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train)
print(f'Hybrid Classical-Quantum HDC(QNN) Train Accuracy: {acc_train * 100:.2f}%')

# Measure inference time
start_inference_time = time.time()
y_pred = sampler_classifier.predict(X_test_proj)
end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Hybrid Classical-Quantum HDC(QNN) Test Accuracy: {accuracy * 100:.2f}%')

# Calculate model memory requirement
model_memory = proj.nbytes + sum([sys.getsizeof(attr) for attr in sampler_classifier.__dict__.values()])

print("Hybrid Classical-Quantum HDC(QNN) Training Time: {:.4f} seconds".format(training_time))
print("Hybrid Classical-Quantum HDC(QNN) Inference Time: {:.4f} seconds".format(inference_time))
print("Hybrid Classical-Quantum HDC(QNN) Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sobol sequence projection...


C:\Users\abuka\AppData\Local\Temp\ipykernel_14172\703812113.py:87: DeprecationWarning: The class ``qiskit.providers.models.backendconfiguration.QasmBackendConfiguration`` is deprecated as of qiskit 1.2. It will be removed in the 2.0 release. The models in ``qiskit.providers.models`` are part of the deprecated `BackendV1` workflow and no longer necessary for `BackendV2`. If a user workflow requires these representations it likely relies on deprecated functionality and should be updated to use `BackendV2`.
  fake_backend = AerSimulator.from_backend(backend)
C:\Users\abuka\AppData\Local\Temp\ipykernel_14172\703812113.py:88: DeprecationWarning: The Sampler and Estimator V1 primitives have been deprecated as of qiskit-ibm-runtime 0.23.0 and will be removed no sooner than 3 months after the release date. Please use the V2 Primitives. See the `V2 migration guide <https://docs.quantum.ibm.com/api/migration-guides/v2-primitives>`_. for more details
  sampler = Sampler(backend=fake_backend)


Hybrid Classical-Quantum HDC(QNN) Train Accuracy: 75.00%
Hybrid Classical-Quantum HDC(QNN) Test Accuracy: 54.55%
Hybrid Classical-Quantum HDC(QNN) Training Time: 273.6750 seconds
Hybrid Classical-Quantum HDC(QNN) Inference Time: 3.0452 seconds
Hybrid Classical-Quantum HDC(QNN) Model Memory Required: 0.0018 MB
